# **RFM ANALYSIS WITH K-MEANS CLUSTERING ALGORITHM**
Recency (R) – How recently a customer made a purchase

Frequency (F) – How often a customer makes purchases

Monetary (M) – How much money a customer spends

# **TASK 1 IMPORT LIBRARIES:**

In [ ]:
import pandas as pd # Pandas is used for data handling, reading files, and working with tables (DataFrames)

import numpy as np # NumPy is used for numerical operations like arrays, math calculations, and statistics

import seaborn as sns

import matplotlib.pyplot as plt # This is used for creating graphs and visualizations like bar charts and scatter plot

from sklearn.preprocessing import StandardScaler # StandardScaler is used to scale data so that it has mean = 0 and standard deviation = 1

from sklearn.cluster import KMeans # KMeans is a clustering algorithm used to group similar data points into clusters

from sklearn.metrics import silhouette_score


This cell imports all necessary Python libraries required for data analysis and clustering.

-Pandas is used for data manipulation.

-NumPy is used for numerical operations.

-Matplotlib is used for visualization.

-StandardScaler is used to normalize the RFM values.

-KMeans is used to perform clustering.

# **TASK 2 LOADING THE DATA SET**

In [ ]:
df = pd.read_excel("Online_Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


This cell loads the Online Retail dataset into a DataFrame.
The dataset contains transactional records including invoice details, customer ID, quantity, price, and purchase date.
The head() function displays the first five rows to understand the structure of the dataset.

In [ ]:
df.shape # number of rows & columns

(541909, 8)

In [ ]:
df.info() # data types & null values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [ ]:
df.describe()  # statistical summary

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


# **EDA PROCESSING**

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64'])

sns.pairplot(num_cols)
plt.show()

In [ ]:
sns.histplot(df["TotalAmount"], kde=True)
plt.title("Sales Distribution")
plt.xlabel("Total Amount")
plt.ylabel("Frequency")
plt.show()

In [ ]:

# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Create subplots
plt.figure(figsize=(15,5))

for i, col in enumerate(num_cols):
    plt.subplot(1, len(num_cols), i+1)   # 1 row, multiple columns
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

# **TASK 3 PREPARING DATA**

In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])


This step removes invalid or incomplete records to ensure accurate analysis.

Rows without CustomerID are removed because customer segmentation requires customer identification.

Rows with negative or zero Quantity and UnitPrice are removed as they represent canceled or invalid transactions.

The InvoiceDate column is converted into datetime format to calculate Recency properly.

# **TASK 4 CREATE A NEW COLUMN**

In [ ]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df['TotalAmount']


This step creates a new column called TotalAmount, calculated as:

Monetary = Quantity × UnitPrice

This represents the total spending per transaction and will be used to compute the Monetary value in RFM analysis.

# **TASK 5 REFERENCE POINT TO CALCULATE R**

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)


The snapshot date is defined as one day after the last transaction date in the dataset. This date acts as a reference point to calculate the Recency, which measures how many days have passed since the customer's last purchase.

# **TASK 6 NORMALISING THE DATA**

In [ ]:
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'TotalAmount': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()


RFM values have different scales (e.g., Recency in days, Monetary in currency).
Since K-Means clustering is distance-based, we normalize the data using StandardScaler.
This ensures all features contribute equally to clustering.

# **TASK 7 SETTING MEAN=0 AND SD=1:**

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)


This cell standardizes the RFM values using StandardScaler.
Standardization transforms the data to have a mean of zero and a standard deviation of one.
This is required because K-Means clustering is distance-based and scaling prevents features with large values from dominating the clustering process.

# **TASK 8 FINDING K VALUE USING WCSS METHOD**

In [ ]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(rfm_scaled)
    wcss.append(kmeans.inertia_)   # inertia_ gives WCSS

plt.figure()
plt.plot(range(1, 11), wcss, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS")
plt.title("Elbow Method for Optimal K")
plt.show()

**From the graph, the curve starts to flatten after K = 4, suggesting that 4 is a suitable choice for customer segmentation. Therefore, the dataset is divided into four meaningful clusters: Loyal Customers, New Customers, Lost Customers, and High-Value Customers**

**Loyal customers --->0**

**New customers ----->1**

**Lost customers ---->2**

**High-value customers->3**

# **TASK 9 APPLING K-MEANS CLUSTERING ALGORITHM**

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)
rfm.head()


This step applies the K-Means algorithm to segment customers into 4 clusters. Each customer is assigned a cluster number based on similar RFM characteristics. The Cluster column is added to the RFM table.

K-Means:

**Takes RFM values of each customer**

**Finds similar customers**

**Groups them into K clusters**

**Each cluster contains customers with similar behavior**

So in your case:
K = 4 (since you have 4 clusters)

It created 4 customer groups automatically

# **TASK 10 FINDING MEAN FOR K_MEANS OF EACH COUSTOMERS**

In [ ]:
cluster_summary = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean()
cluster_summary




# **TASK 10 CREATING HEATMAP**

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(
    cluster_summary,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu"
)

plt.title("RFM Cluster Heatmap")
plt.show()

Cluster 2 –Best / VIP Customers.

Recency = 4.7 (Very low → purchased recently).

Frequency = 2565.3 (Very high → purchase very often).

Monetary = 126118.3 (Extremely high → spend a lot).

These are your most valuable customers.

They:

Buy recently.

Buy frequently.

Spend the most.


# **TASK 11 FINDING RFM VALUES**

In [ ]:
rfm.groupby('Cluster').mean()


This step calculates the average RFM values for each cluster. By analyzing these averages, we can label the clusters as:

Loyal Customers --------->0.

Occasional / Average Customers----->1.

VIP / High-Value Customers ---->2.

Lost / Churn Customers-------->3.

This provides meaningful business insights for marketing strategies.

# **TASK 12 FINDING RFM VALUES**

In [ ]:
plt.figure()
plt.scatter(rfm['Recency'], rfm['Monetary'], c=rfm['Cluster'])
plt.xlabel("Recency")
plt.ylabel("Monetary")
plt.title("Customer Segmentation using RFM + KMeans")
plt.show()

This step creates a scatter plot to visualize customer segmentation.

🟢 Green Points at the Top (Very High Monetary).

🟣 Low Recency + Low/Medium Monetary (Left Bottom Area)Active but Low-Spending Customers.

🔵 Medium Recency + Medium Spending (Center Area)-Moderate spenders.

🟡 High Recency + Low Monetary (Right Side)These are Lost / Inactive Customers.